<a href="https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Content Results Curve

The paper reports that content health is highest around the 61–90 day age range and is lower in older age buckets. The 61–90 day group has a Health Score of 37.2, while the 271–365 day group has a Health Score of 29.6. The paper also notes that the 365+ group still has a relatively high average Health Score, so older content is not automatically unsuccessful.

**Methodology question:** I would first ask how the outcome is defined and whether the age buckets are directly comparable. Health Score is a FlyRank composite metric, while content age is the grouping variable. The observed difference between age groups supports an association, but it does not by itself prove that aging causes the decline. Different content mixes, clients, or historical performance could contribute to the difference.

I would therefore ask whether a time-aware analysis following the same pages over time, or controls for major confounding factors, would support the stronger lifecycle interpretation. This is a constructive validation question rather than a criticism of the finding.

### Finding 2 — The Freshness Multiplier

The paper reports that the 31–90 day freshness window has the strongest stable growth-to-decline ratio at 5.43:1. It also reports a separate comparison for pages older than 365 days, where recently refreshed pages had higher Health Score and much higher impressions than pages that had not been refreshed recently.

**Methodology question:** I would ask how the refreshed-versus-stale comparison defines the outcome and whether the groups were comparable before the refresh. A page may have been selected for refreshing because it already had stronger visibility, strategic value, or a different performance history. Therefore, the observed higher health and impressions after refresh do not by themselves establish that refreshing caused the improvement.

A stronger causal claim would require a suitable before/after design with a comparable control group, or an explicit experiment. For the current evidence, I would describe the result as an observed association or directional signal rather than proof that refreshing causes the reported lift.

In [1]:
# W06 — Paper finding evidence checks

paper_findings = {
    "Finding 1 - Content Results Curve": {
        "61-90_day_health": 37.2,
        "271-365_day_health": 29.6
    },
    "Finding 2 - Freshness Multiplier": {
        "31-90_growth_decline_ratio": 5.43,
        "365_plus_health_refreshed": 37,
        "365_plus_health_comparison": 23,
        "365_plus_impression_lift": 52
    }
}

print("Paper finding checks")
print("--------------------")

print(
    "Finding 1: 61-90 day Health Score =",
    paper_findings["Finding 1 - Content Results Curve"]["61-90_day_health"]
)

print(
    "Finding 1: 271-365 day Health Score =",
    paper_findings["Finding 1 - Content Results Curve"]["271-365_day_health"]
)

print(
    "Finding 2: 31-90 day growth:decline ratio =",
    paper_findings["Finding 2 - Freshness Multiplier"]["31-90_growth_decline_ratio"]
)

print(
    "Finding 2: reported 365+ impression lift =",
    paper_findings["Finding 2 - Freshness Multiplier"]["365_plus_impression_lift"],
    "x"
)

Paper finding checks
--------------------
Finding 1: 61-90 day Health Score = 37.2
Finding 1: 271-365 day Health Score = 29.6
Finding 2: 31-90 day growth:decline ratio = 5.43
Finding 2: reported 365+ impression lift = 52 x


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I re-ran my Random Forest model using the same final feature set:

* `impressions_90d`
* `sessions_90d`
* `content_age_days`
* `ctr`
* `avg_position`
* `word_count`
* `engagement_rate`

A total of **22,301 rows** were available for validation.

### Before — Row-level split

With a standard row-level train/test split, the model achieved a **Precision@50 of 0.92**.

However, there was **client overlap between the training and test sets: 32 clients**. This means that pages from the same clients could appear in both training and testing. The model could therefore benefit from client-specific patterns that were already present during training.

### After — Client-grouped split

I then used a client-grouped split, keeping entire clients in either the training set or the test set. This resulted in:

* **25 training clients**
* **7 test clients**
* **19,489 training rows**
* **2,812 test rows**
* **0 client overlap**
* **Precision@50 of 0.84**

The Precision@50 decreased from **0.92 to 0.84** after applying the grouped validation design. This decrease is expected because the grouped split provides a more honest test of whether the model can rank pages from clients it has not seen during training.

The comparison shows that the row-level result was likely optimistic because the model had exposure to the same clients in both training and testing. The client-grouped result is therefore the more appropriate number to use when discussing generalization to unseen clients.

**Interpretation:** The model still achieved a measured Precision@50 of **0.84** under the honest client-grouped split, which suggests useful ranking performance on the evaluated anonymized test set. However, this is directional evidence for decision-support, not proof that the model can predict future content decline or that a page definitely needs a refresh.

The validation guide recommends client/group holdout when pages from the same client may share patterns, because the test should represent clients the model has not effectively already seen.


In [2]:
# W06 — Honest validation audit
# Compare a row-level split with a client-grouped split.

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


# ---------------------------------------------------------
# 1. Load the starter dataset
# ---------------------------------------------------------

data_path = Path("data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    !git clone -q https://github.com/Anshikaag-28/Anshika-FlyRank-ML.git /content/Anshika-FlyRank-ML
    %cd /content/Anshika-FlyRank-ML
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)


# ---------------------------------------------------------
# 2. Create the Week-5 proxy target
# ---------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)


# ---------------------------------------------------------
# 3. Use the same Week-5 feature set
# ---------------------------------------------------------

features = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "ctr",
    "avg_position",
    "word_count",
    "engagement_rate"
]

features = [c for c in features if c in df.columns]

required_columns = features + [
    "is_declining_label",
    "client_id"
]

model_df = df.dropna(
    subset=required_columns
).copy()

print("Rows available for validation:", len(model_df))
print("Features:", features)


# ---------------------------------------------------------
# 4. Precision@50
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k=50):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    ranking = np.argsort(scores)[::-1][:k]

    return y_true[ranking].mean()


# ---------------------------------------------------------
# 5. Function to train Random Forest and evaluate it
# ---------------------------------------------------------

def train_and_score(X_train, y_train, X_test, y_test):

    # Fill missing values using training medians
    train_medians = X_train.median(numeric_only=True)

    X_train = X_train.fillna(train_medians)
    X_test = X_test.fillna(train_medians)

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    model.fit(X_train, y_train)

    scores = model.predict_proba(X_test)[:, 1]

    precision_50 = precision_at_k(
        y_test,
        scores,
        k=50
    )

    return precision_50


# =========================================================
# BEFORE — ordinary row-level split
# =========================================================

row_train, row_test = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42,
    stratify=model_df["is_declining_label"]
)

row_precision_50 = train_and_score(
    row_train[features],
    row_train["is_declining_label"],
    row_test[features],
    row_test["is_declining_label"]
)


# Check how many clients appear in both sets
row_client_overlap = set(
    row_train["client_id"]
).intersection(
    set(row_test["client_id"])
)

print("\nBEFORE — Row-level split")
print("------------------------")
print("Training rows:", len(row_train))
print("Test rows:", len(row_test))
print("Client overlap:", len(row_client_overlap))
print("Precision@50:", round(row_precision_50, 3))


# =========================================================
# AFTER — client-grouped split
# =========================================================

clients = model_df["client_id"].unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

group_train = model_df[
    model_df["client_id"].isin(train_clients)
].copy()

group_test = model_df[
    model_df["client_id"].isin(test_clients)
].copy()

group_precision_50 = train_and_score(
    group_train[features],
    group_train["is_declining_label"],
    group_test[features],
    group_test["is_declining_label"]
)

group_client_overlap = set(
    group_train["client_id"]
).intersection(
    set(group_test["client_id"])
)

print("\nAFTER — Client-grouped split")
print("-----------------------------")
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Training rows:", len(group_train))
print("Test rows:", len(group_test))
print("Client overlap:", len(group_client_overlap))
print("Precision@50:", round(group_precision_50, 3))


# ---------------------------------------------------------
# Before vs After comparison
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "Validation design": [
        "Row-level split",
        "Client-grouped split"
    ],
    "Precision@50": [
        row_precision_50,
        group_precision_50
    ],
    "Client overlap": [
        len(row_client_overlap),
        len(group_client_overlap)
    ]
})

print("\nBefore vs After")
display(comparison)

/content/Anshika-FlyRank-ML
Rows available for validation: 22301
Features: ['impressions_90d', 'sessions_90d', 'content_age_days', 'ctr', 'avg_position', 'word_count', 'engagement_rate']

BEFORE — Row-level split
------------------------
Training rows: 17840
Test rows: 4461
Client overlap: 32
Precision@50: 0.92

AFTER — Client-grouped split
-----------------------------
Training clients: 25
Test clients: 7
Training rows: 19489
Test rows: 2812
Client overlap: 0
Precision@50: 0.84

Before vs After


,Validation design,Precision@50,Client overlap
0,Row-level split,0.92,32
1,Client-grouped split,0.84,0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I checked the final Week-5 feature set against target-derived fields and FlyRank product-decision fields. `trend_direction`, `trend_pct`, `is_declining_label`, `priority_score`, `action_type`, `health_score`, and `refresh_tier` were treated as forbidden model features.

I also checked that the client-grouped training and test sets have no client overlap.

The audit passed if the forbidden-feature intersection is empty and the grouped train/test client intersection is empty. This reduces obvious feature and validation leakage, although it does not prove that every possible temporal or measurement bias has been eliminated.

In [3]:
# W06 — Leakage audit

forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "priority_score",
    "action_type",
    "health_score",
    "refresh_tier"
}

used_features = set(features)

leaked_features = used_features.intersection(
    forbidden_features
)

print("Final model features:")
for feature in features:
    print("-", feature)

print("\nForbidden features detected:")
print(leaked_features)

assert len(leaked_features) == 0

print("\nPASS: No forbidden target/product-decision fields are used as model features.")


# ---------------------------------------------------------
# Check client-grouped split
# ---------------------------------------------------------

assert set(group_train["client_id"]).isdisjoint(
    set(group_test["client_id"])
)

print("PASS: No client appears in both grouped train and test sets.")


# ---------------------------------------------------------
# Check target is not a feature
# ---------------------------------------------------------

assert "is_declining_label" not in features

print("PASS: Target is not included as a model feature.")


# ---------------------------------------------------------
# Check trend fields are not features
# ---------------------------------------------------------

assert "trend_direction" not in features
assert "trend_pct" not in features

print("PASS: Trend-derived fields are excluded from model features.")

Final model features:
- impressions_90d
- sessions_90d
- content_age_days
- ctr
- avg_position
- word_count
- engagement_rate

Forbidden features detected:
set()

PASS: No forbidden target/product-decision fields are used as model features.
PASS: No client appears in both grouped train and test sets.
PASS: Target is not included as a model feature.
PASS: Trend-derived fields are excluded from model features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

"My Random Forest predicts which pages need to be refreshed and performs better than the Week-4 baseline."

### Safer claim

"On the evaluated anonymized test split, the Random Forest achieved a higher measured Precision@50 than the Week-4 baseline for ranking pages associated with the observed declining proxy. This is directional evidence that the model can support prioritization of pages for human review. It does not establish that the model predicts future decline, that a page definitely needs a refresh, or that refreshing a page will cause performance to improve."

In [4]:
# W06 — Claim safety check

original_claim = (
    "The Random Forest predicts which pages need to be refreshed "
    "and performs better than the Week-4 baseline."
)

safer_claim = (
    "On the evaluated anonymized test split, the Random Forest "
    "achieved a higher measured Precision@50 than the Week-4 baseline "
    "for ranking pages associated with the observed declining proxy. "
    "This is directional evidence for human review prioritization, "
    "not proof of future decline or proof that refreshing a page "
    "will cause performance to improve."
)

print("Original claim:")
print(original_claim)

print("\nSafer claim:")
print(safer_claim)

# Simple checks for safer language
required_terms = [
    "measured",
    "observed",
    "directional",
    "human review"
]

for term in required_terms:
    assert term.lower() in safer_claim.lower()

print("\nPASS: Safer claim uses evidence-limited language.")

Original claim:
The Random Forest predicts which pages need to be refreshed and performs better than the Week-4 baseline.

Safer claim:
On the evaluated anonymized test split, the Random Forest achieved a higher measured Precision@50 than the Week-4 baseline for ranking pages associated with the observed declining proxy. This is directional evidence for human review prioritization, not proof of future decline or proof that refreshing a page will cause performance to improve.

PASS: Safer claim uses evidence-limited language.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.